In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [2]:
import numpy as np
import pandas as pd
from scripts.models.rolling_hmm import full_rolling_hmm_portfolio  # your function

def live_next_weights(features_fresh, asset_returns_fresh, window=272, step=21):
    """
    EXTENDS your full_rolling_hmm_portfolio() for NEXT prediction
    """
    # STEP 1: Run YOUR exact function on fresh data
    weights_fresh = full_rolling_hmm_portfolio(
        features_df=features_fresh, 
        asset_returns=asset_returns_fresh, 
        window=window, 
        step=step
    )
    
    # STEP 2: Get LAST weights + regime (forward prediction)
    latest_date = weights_fresh.index[-1]
    latest_weights = weights_fresh.iloc[-1].drop('regime')  # exclude regime col
    latest_regime = weights_fresh['regime'].iloc[-1]
    
    print(f"🎯 NEXT REBALANCE: {latest_date + pd.Timedelta(days=step)}")
    print(f"📊 Regime: {latest_regime}, Weights:\n{latest_weights.round(3)}")
    
    return latest_weights, latest_regime

In [5]:
#!/usr/bin/env python3
"""
BROKER → TABLE (Correct Portfolio)
8 Orders → Perfect $1M HMM positions!
"""

def broker_to_table_orders():
    """Current broker fills → Your CORRECT table positions"""
    
    # BROKER CURRENT (Screenshot 1)
    broker_positions = {
        'TAIL': 5267,   'SPY': 3232,  'EFA': 3202,
        'USMV': -1,     'NVDA': -140,  'META': -167,
        'VNQ': -1969, 'VIXY':-77, 'AMZN':-146, 'AAPL':799
    }
    
    # TABLE TARGETS (CORRECT - Screenshot 2)
    table_targets = {
        'TAIL': 41353,  'SPY': 1013,  'EFA': 594,
        'USMV': -348,   'NVDA': 0,    'META': -78,
        'VNQ': -1749,   'AAPL': 39,   'AMZN': -38,
        'VIXY': 444
    }
    
    # TABLE PRICES (for limits/stops)
    prices = {
        'AAPL': 273.08, 'AMZN': 232.53, 'EFA': 96.44, 
        'META': 665.95, 'NVDA': 187.54, 'SPY': 687.01,
        'TAIL': 11.49,  'USMV': 94.95,  'VIXY': 25.47,
        'VNQ': 89.22
    }
    
    print("🎯 BROKER → TABLE ORDERS (Correct $1M Portfolio)")
    print("="*80)
    
    orders = []
    all_tickers = set(broker_positions) | set(table_targets)
    
    for ticker in sorted(all_tickers):
        current = broker_positions.get(ticker, 0)
        target = table_targets.get(ticker, 0)
        trade_shares = target - current
        
        if trade_shares != 0:
            price = prices.get(ticker, 100)
            
            if trade_shares > 0:  # BUY
                limit = round(price * 1.005, 2)
                stop = round(price * 0.85, 2)
                action = "BUY"
            else:  # SELL
                limit = round(price * 0.995, 2)
                stop = round(price * 1.15, 2)
                action = "SELL"
            
            print(f"{action:>5} {ticker:>5} | {trade_shares:>+7.0f} shares")
            print(f"   ({current:>+5.0f} → {target:>+5.0f}) | Current: ${price:>6.2f}")
            print(f"   📋 ORDER: {trade_shares:+7.0f} {ticker} LMT ${limit:.2f} STP ${stop:.2f}")
            print()
            
            orders.append({
                'ticker': ticker, 'action': action, 'shares': trade_shares,
                'current': current, 'target': target, 'limit': limit, 'stop': stop
            })
    
    orders_df = pd.DataFrame(orders)
    print(f"✅ {len(orders_df)} ORDERS → TABLE PORTFOLIO PERFECT!")
    return orders_df

# ✅ RUN NOW
table_orders = broker_to_table_orders()
table_orders.to_csv("../data/processed/broker_to_table_orders.csv", index=False)



🎯 BROKER → TABLE ORDERS (Correct $1M Portfolio)
 SELL  AAPL |    -760 shares
   ( +799 →   +39) | Current: $273.08
   📋 ORDER:    -760 AAPL LMT $271.71 STP $314.04

  BUY  AMZN |    +108 shares
   ( -146 →   -38) | Current: $232.53
   📋 ORDER:    +108 AMZN LMT $233.69 STP $197.65

 SELL   EFA |   -2608 shares
   (+3202 →  +594) | Current: $ 96.44
   📋 ORDER:   -2608 EFA LMT $95.96 STP $110.91

  BUY  META |     +89 shares
   ( -167 →   -78) | Current: $665.95
   📋 ORDER:     +89 META LMT $669.28 STP $566.06

  BUY  NVDA |    +140 shares
   ( -140 →    +0) | Current: $187.54
   📋 ORDER:    +140 NVDA LMT $188.48 STP $159.41

 SELL   SPY |   -2219 shares
   (+3232 → +1013) | Current: $687.01
   📋 ORDER:   -2219 SPY LMT $683.57 STP $790.06

  BUY  TAIL |  +36086 shares
   (+5267 → +41353) | Current: $ 11.49
   📋 ORDER:  +36086 TAIL LMT $11.55 STP $9.77

 SELL  USMV |    -347 shares
   (   -1 →  -348) | Current: $ 94.95
   📋 ORDER:    -347 USMV LMT $94.48 STP $109.19

  BUY  VIXY |    +521 

In [3]:
# RUN YOUR EXISTING PIPELINE with fresh data
features_fresh = pd.read_csv('../data/processed/selected_feature_matrix.csv', parse_dates=True, index_col=0)
asset_returns_fresh = features_fresh[[col for col in features_fresh.columns if col.endswith('_ret')]]

# Then predict
next_weights, regime = live_next_weights(features_fresh.tail(500), asset_returns_fresh.tail(500))

🚀 Rolling (drop NaN regimes): T=500, window=272
AAPL: final log-likelihood = -762.8736
AMZN: final log-likelihood = -641.5281
EFA: final log-likelihood = -642.8186
META: final log-likelihood = -514.8842
NVDA: final log-likelihood = -581.9189
SPY: final log-likelihood = -760.7635
TAIL: final log-likelihood = -545.7824
USMV: final log-likelihood = -639.1345
VIXY: final log-likelihood = -538.0213
VNQ: final log-likelihood = -654.6501
t=272: 272 raw → 272 valid regimes (100%)
  Regime 0: 271 days ✓
  Regime 1: only 1 days → equal weights
AAPL: final log-likelihood = -763.6652
AMZN: final log-likelihood = -655.6626
EFA: final log-likelihood = -761.2974
META: final log-likelihood = -529.6754


Model is not converging.  Current: -545.8064109429686 is not greater than -545.8064076332175. Delta is -3.3097510367952054e-06
Model is not converging.  Current: -538.5673638489694 is not greater than -538.5673487344596. Delta is -1.5114509778868523e-05


NVDA: final log-likelihood = -596.4353
SPY: final log-likelihood = -663.4542
TAIL: final log-likelihood = -545.8064
USMV: final log-likelihood = -643.3453
VIXY: final log-likelihood = -538.5674
VNQ: final log-likelihood = -638.7551
t=293: 272 raw → 272 valid regimes (100%)
  Regime 0: 260 days ✓
  Regime 1: only 12 days → equal weights
AAPL: final log-likelihood = -627.2906
AMZN: final log-likelihood = -692.4325
EFA: final log-likelihood = -763.4391
META: final log-likelihood = -588.4827
NVDA: final log-likelihood = -601.2032
SPY: final log-likelihood = -755.8750
TAIL: final log-likelihood = -531.0027
USMV: final log-likelihood = -626.6800
VIXY: final log-likelihood = -559.2074
VNQ: final log-likelihood = -760.5231
t=314: 272 raw → 272 valid regimes (100%)
  Regime 0: 136 days ✓
  Regime 1: 136 days ✓
AAPL: final log-likelihood = -519.1653
AMZN: final log-likelihood = -575.9002


Model is not converging.  Current: -513.0295610026524 is not greater than -513.0295594234843. Delta is -1.579168042553647e-06


EFA: final log-likelihood = -467.7924
META: final log-likelihood = -530.6889
NVDA: final log-likelihood = -593.2881
SPY: final log-likelihood = -472.3876
TAIL: final log-likelihood = -503.9142
USMV: final log-likelihood = -403.1725
VIXY: final log-likelihood = -515.9548
VNQ: final log-likelihood = -610.8030
t=335: 272 raw → 272 valid regimes (100%)
  Regime 0: 255 days ✓
  Regime 1: only 17 days → equal weights
AAPL: final log-likelihood = -513.0296
AMZN: final log-likelihood = -585.1160
EFA: final log-likelihood = -468.8702
META: final log-likelihood = -548.8113
NVDA: final log-likelihood = -584.6433
SPY: final log-likelihood = -487.9732
TAIL: final log-likelihood = -381.8339
USMV: final log-likelihood = -448.8716
VIXY: final log-likelihood = -541.8561
VNQ: final log-likelihood = -599.0462
t=356: 272 raw → 272 valid regimes (100%)
  Regime 0: 248 days ✓
  Regime 1: 24 days ✓
AAPL: final log-likelihood = -510.7598
AMZN: final log-likelihood = -588.2020
EFA: final log-likelihood = -468.

Model is not converging.  Current: -498.75955498645556 is not greater than -498.75955301675083. Delta is -1.969704726434429e-06
Model is not converging.  Current: -503.80230755437105 is not greater than -503.80230541827643. Delta is -2.1360946220738697e-06
Model is not converging.  Current: -505.13699028181753 is not greater than -505.13698810106644. Delta is -2.1807510961480148e-06


AAPL: final log-likelihood = -498.7596
AMZN: final log-likelihood = -571.1952
EFA: final log-likelihood = -470.3172
META: final log-likelihood = -547.7168
NVDA: final log-likelihood = -535.5770
SPY: final log-likelihood = -489.6223
TAIL: final log-likelihood = -378.0048
USMV: final log-likelihood = -446.5169
VIXY: final log-likelihood = -539.1200
VNQ: final log-likelihood = -595.2893
t=398: 272 raw → 272 valid regimes (100%)
  Regime 0: 218 days ✓
  Regime 1: 54 days ✓
AAPL: final log-likelihood = -503.8023
AMZN: final log-likelihood = -610.4991
EFA: final log-likelihood = -466.3499
META: final log-likelihood = -543.5221
NVDA: final log-likelihood = -506.8917
SPY: final log-likelihood = -477.3382
TAIL: final log-likelihood = -368.7340
USMV: final log-likelihood = -414.0569
VIXY: final log-likelihood = -521.5600
VNQ: final log-likelihood = -591.0864
t=419: 272 raw → 272 valid regimes (100%)
  Regime 0: 242 days ✓
  Regime 1: 30 days ✓
AAPL: final log-likelihood = -505.1370
AMZN: final l

Model is not converging.  Current: -505.9669535800255 is not greater than -505.9669514671. Delta is -2.1129255287632986e-06
Model is not converging.  Current: -395.2120153950414 is not greater than -395.21201011366014. Delta is -5.281381277200126e-06


VNQ: final log-likelihood = -584.4064
t=440: 272 raw → 272 valid regimes (100%)
  Regime 0: 248 days ✓
  Regime 1: 24 days ✓
AAPL: final log-likelihood = -501.3595
AMZN: final log-likelihood = -591.8376
EFA: final log-likelihood = -400.5492
META: final log-likelihood = -517.9134
NVDA: final log-likelihood = -481.8628
SPY: final log-likelihood = -438.8096
TAIL: final log-likelihood = -415.7100
USMV: final log-likelihood = -423.3670
VIXY: final log-likelihood = -586.1077
VNQ: final log-likelihood = -584.2453
t=461: 272 raw → 272 valid regimes (100%)
  Regime 0: 249 days ✓
  Regime 1: 23 days ✓
AAPL: final log-likelihood = -505.9670
AMZN: final log-likelihood = -602.1857
EFA: final log-likelihood = -395.2120
META: final log-likelihood = -537.6302
NVDA: final log-likelihood = -500.2515
SPY: final log-likelihood = -448.8040
TAIL: final log-likelihood = -330.3221
USMV: final log-likelihood = -418.2343
VIXY: final log-likelihood = -596.7976
VNQ: final log-likelihood = -579.2455
t=482: 272 raw

In [4]:
def transition_trades(prev_weights, curr_weights):
    """Auto-detects real portfolio value"""
    all_assets = sorted(set(prev_weights.index) | set(curr_weights.index))
    prev_w = prev_weights.reindex(all_assets).fillna(0)
    curr_w = curr_weights.reindex(all_assets).fillna(0)
    
    # DETECT real portfolio value from prev holdings
    portfolio_value = abs(prev_w[prev_w != 0]).sum() * 10000  # Scale to match your data
    
    prev_value = portfolio_value * prev_w
    target_value = portfolio_value * curr_w
    trades = pd.DataFrame({
        'prev_weight': prev_w,
        'curr_weight': curr_w.round(3),
        'prev_value': prev_value.round(0),
        'target_value': target_value.round(0),
        'trade_size': (target_value - prev_value).round(0)
    })
    
    trades['action'] = trades['trade_size'].apply(
        lambda x: 'BUY' if x > 10 else 'SELL' if x < -10 else 'HOLD'
    )
    trades['shares_estimate'] = (trades['trade_size'] / 100).round(0)
    
    total_turnover = trades['trade_size'].abs().sum() / portfolio_value
    print(f"✅ Portfolio: ${portfolio_value:,.0f} | Turnover: {total_turnover:.1%}")
    
    return trades

# === USAGE ===

weights_rolling = pd.read_csv("../data/processed/portfolio_hmm_utils_rolling.csv", index_col=0, parse_dates=True)
prev_weights = weights_rolling.iloc[-1].drop('regime')
curr_weights = next_weights 

trades = transition_trades(prev_weights, curr_weights)
print("\n🎯 TRADE ORDERS:")
print(trades[trades['action'] != 'HOLD'].round(0))


✅ Portfolio: $15,000 | Turnover: 119.3%

🎯 TRADE ORDERS:
      prev_weight  curr_weight  prev_value  target_value  trade_size action  \
AAPL         -0.0          0.0        -4.0          56.0        60.0    BUY   
AMZN         -0.0         -0.0     -2314.0         -22.0      2292.0    BUY   
EFA          -0.0          0.0        -0.0        2455.0      2455.0    BUY   
META         -0.0         -0.0      -952.0        -516.0       436.0    BUY   
NVDA          0.0         -0.0      2656.0        -135.0     -2791.0   SELL   
SPY           1.0          1.0      9498.0        8730.0      -768.0   SELL   
TAIL          0.0          1.0      4288.0        7509.0      3220.0    BUY   
TSLA          0.0          0.0      1253.0           0.0     -1253.0   SELL   
USMV         -0.0         -0.0      -481.0          -0.0       481.0    BUY   
VIXY          0.0          0.0      1054.0           0.0     -1054.0   SELL   
VNQ           0.0         -0.0         0.0       -3078.0     -3078.0   SEL

In [5]:
asset_ret_df = pd.read_csv("../data/processed/feature_matrix.csv")
latest_date = asset_ret_df.iloc[-1].Date
asset_ret_df.set_index('Date', inplace=True)

curr_regime = pd.Series(regime, name="regime", index=[latest_date])
curr_weights_df = pd.DataFrame(curr_weights).T
curr_weights_df.index = [latest_date]
curr_weights_df = pd.concat([curr_weights_df,curr_regime], axis = 1)


In [6]:
def log_trades(weights_df, filepath="../data/processed/actual_trades_log.csv"):
    """
    If filepath doesn't exist: CREATE + write first row
    If exists: APPEND new row
    """
    # Create directory if missing
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    
    if not os.path.exists(filepath):
        # FIRST TIME: Create with current weights
        print(f"✅ Creating {filepath}")
        curr_weights_df.to_csv(filepath)
        print("✅ First weights written!")
    else:
        # APPEND MODE: Read → add row → save
        print(f"📝 Appending to {filepath}")
        existing = pd.read_csv(filepath, index_col=0, parse_dates=True)
        existing = pd.concat([existing,curr_weights_df], axis = 0)
        existing.to_csv(filepath)
        print("✅ Row appended!")
    
    return pd.read_csv(filepath, index_col=0, parse_dates=True)

In [7]:
weights_df = log_trades(curr_weights_df, filepath="../data/processed/actual_trades_log.csv")

📝 Appending to ../data/processed/actual_trades_log.csv
✅ Row appended!


In [8]:
asset_ret_df

,AAPL_ret,AMZN_ret,BIL_ret,BND_ret,COIN_ret,DBC_ret,DIA_ret,EEM_ret,EFA_ret,ETHA_ret,...,XLB_mom,XLE_mom,XLF_mom,XLI_mom,XLK_mom,XLP_mom,XLRE_mom,XLU_mom,XLV_mom,XLY_mom
Date,,,,,,,,,,,,,,,,,,,,,
2015-01-02,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-05,-0.028576,-0.020731,0.000219,0.002899,0.000000,-0.014365,-0.017576,-0.017957,-0.023888,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-06,0.000094,-0.023098,-0.000219,0.002891,0.000000,-0.009505,-0.008328,-0.004211,-0.011392,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-07,0.013925,0.010544,0.000000,0.000601,0.000000,-0.006199,0.012609,0.021394,0.011054,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-08,0.037703,0.006813,0.000000,-0.001564,0.000000,0.003949,0.017892,0.016893,0.013439,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-22,-0.009915,0.004739,0.000000,-0.000270,0.011278,0.012765,0.004790,0.005384,0.002511,-0.007092,...,0.063248,-0.002754,0.074383,0.053184,0.064277,0.007764,-0.006620,-0.029196,0.008740,0.087206
2025-12-23,0.005117,0.016111,0.000000,-0.000135,-0.022849,0.011104,0.001591,0.005539,0.006146,-0.002673,...,0.063018,0.006602,0.071955,0.050185,0.045164,0.014699,-0.008322,-0.036696,0.002512,0.071394
2025-12-24,0.005310,0.001033,0.000329,0.002429,-0.010663,-0.000442,0.005725,0.002023,0.001245,-0.011213,...,0.049462,0.010004,0.064423,0.040281,0.044868,0.008798,-0.009882,-0.028697,-0.014528,0.052553


In [9]:
weights_df

,AAPL,AMZN,EFA,META,NVDA,SPY,TAIL,USMV,VIXY,VNQ,regime
2025-12-26 00:00:00,0.089058,-4.152334e-07,-4.757324e-07,-0.066185,-0.039550,0.644451,0.514667,-3.404807e-02,1.824230e-03,-0.110217,1
2025-12-29,0.003757,-1.437777e-03,1.636853e-01,-0.034389,-0.008987,0.581989,0.500569,-2.137624e-08,1.935913e-07,-0.205186,0


In [10]:
common_dates = weights_df.index.intersection(asset_ret_df.index)

In [11]:
common_dates

Index(['2025-12-29'], dtype='object')

In [12]:
def compute_portfolio_returns(weights_df, asset_ret_df):
    """
    weights_df: DataFrame of portfolio weights (dates × assets), no 'regime' column
    asset_ret_df: DataFrame of asset returns aligned by date (same asset columns)
                  e.g. daily log or simple returns
    """
    asset_ret_df.columns = [col.replace('_ret','') if col.endswith('_ret') else col for col in asset_ret_df.columns]
    
    # Align dates & assets
    common_dates = weights_df.index.intersection(asset_ret_df.index)
    common_assets = weights_df.columns.intersection(asset_ret_df.columns)
    w = weights_df.loc[common_dates, common_assets]
    r = asset_ret_df.loc[common_dates, common_assets]

    # Portfolio return each day: sum_i w_{t-1,i} * r_{t,i}
    print(w)
    print(len(w))
    w_shift = w.shift(1).fillna(0) if len(w) > 1 else 0
    port_ret = (w_shift * r).sum(axis=1)
    port_ret.name = "portfolio_return"
    return port_ret

def summarize_performance(port_ret, rf_annual=0.0, periods_per_year=252):
    """
    port_ret: Series of periodic portfolio returns (e.g. daily)
    rf_annual: annual risk-free rate (as decimal)
    """
    rf_periodic = (1 + rf_annual)**(1/periods_per_year) - 1

    # Basic stats
    mean_ret = port_ret.mean()
    vol = port_ret.std()
    ann_ret = (1 + mean_ret)**periods_per_year - 1
    ann_vol = vol * np.sqrt(periods_per_year)

    # Sharpe
    excess = port_ret - rf_periodic
    sharpe = np.nan
    if excess.std() > 0:
        sharpe = (excess.mean() / excess.std()) * np.sqrt(periods_per_year)

    # Max drawdown
    cum = (1 + port_ret).cumprod()
    peak = cum.cummax()
    dd = (cum / peak) - 1
    max_dd = dd.min()

    return {
        "mean_daily_ret": mean_ret,
        "daily_vol": vol,
        "ann_return": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": max_dd
    }


In [13]:
"""
# Create from your features_df (last 7 days for paper test)
features_df = pd.read_csv("../data/processed/selected_feature_matrix.csv", index_col=0, parse_dates=True)
asset_ret = features_df.filter(like='_ret').tail(7)  # Last 7 days returns

asset_ret.to_csv("../data/processed/live_asset_returns.csv")
print("✅ live_asset_returns.csv created (7 days)")
print(asset_ret.tail())

# Your paper trades become "live history"
curr_weights = pd.read_csv("../data/processed/portfolio_hmm_utils_rolling.csv", index_col=0, parse_dates=True).iloc[-1].drop('regime')
# Create 1-row history (your Dec 29 paper trade)
weights_history = pd.DataFrame([curr_weights]).T  # Transpose to assets × dates
weights_history.columns = [pd.Timestamp.today()]  # Your paper trade date
weights_history.to_csv("../data/processed/current_portfolio_weights_history.csv")

print("✅ current_portfolio_weights_history.csv created")
print(weights_history)

# NOW works instantly
live_weights = pd.read_csv("../data/processed/current_portfolio_weights_history.csv", index_col=0, parse_dates=True).T
asset_ret = pd.read_csv("../data/processed/live_asset_returns.csv", index_col=0, parse_dates=True)
"""

'\n# Create from your features_df (last 7 days for paper test)\nfeatures_df = pd.read_csv("../data/processed/selected_feature_matrix.csv", index_col=0, parse_dates=True)\nasset_ret = features_df.filter(like=\'_ret\').tail(7)  # Last 7 days returns\n\nasset_ret.to_csv("../data/processed/live_asset_returns.csv")\nprint("✅ live_asset_returns.csv created (7 days)")\nprint(asset_ret.tail())\n\n# Your paper trades become "live history"\ncurr_weights = pd.read_csv("../data/processed/portfolio_hmm_utils_rolling.csv", index_col=0, parse_dates=True).iloc[-1].drop(\'regime\')\n# Create 1-row history (your Dec 29 paper trade)\nweights_history = pd.DataFrame([curr_weights]).T  # Transpose to assets × dates\nweights_history.columns = [pd.Timestamp.today()]  # Your paper trade date\nweights_history.to_csv("../data/processed/current_portfolio_weights_history.csv")\n\nprint("✅ current_portfolio_weights_history.csv created")\nprint(weights_history)\n\n# NOW works instantly\nlive_weights = pd.read_csv(".

In [14]:
port_ret = compute_portfolio_returns(weights_df,asset_ret_df)
print(port_ret)
stats = summarize_performance(port_ret)

print("LIVE PAPER PERFORMANCE:")
for k, v in stats.items():
    print(f"{k}: {v:.4f}")


                AAPL      AMZN       EFA      META      NVDA       SPY  \
2025-12-29  0.003757 -0.001438  0.163685 -0.034389 -0.008987  0.581989   

                TAIL          USMV          VIXY       VNQ  
2025-12-29  0.500569 -2.137624e-08  1.935913e-07 -0.205186  
1
2025-12-29    0.0
Name: portfolio_return, dtype: float64
LIVE PAPER PERFORMANCE:
mean_daily_ret: 0.0000
daily_vol: nan
ann_return: 0.0000
ann_vol: nan
sharpe: nan
max_drawdown: 0.0000


In [15]:
"""
def update_live_performance():
    """Run DAILY after paper/live trades"""
    # Append today's weights
    new_weights = full_rolling_hmm_portfolio(fresh_data.tail(500)).iloc[-1].drop('regime')
    weights_history[new_weights.name] = new_weights
    
    # Append today's returns  
    today_ret = fetch_live_returns()
    asset_ret = pd.concat([asset_ret, today_ret])
    
    # Save & compute
    weights_history.to_csv("../data/processed/current_portfolio_weights_history.csv")
    asset_ret.to_csv("../data/processed/live_asset_returns.csv")
    
    stats = summarize_performance(compute_portfolio_returns(weights_history.T, asset_ret))
    print(f"LIVE SHARPE: {stats['sharpe']:.2f}")

update_live_performance()
"""

SyntaxError: invalid syntax (2890922507.py, line 3)